In [1]:
# Load the SQL toolset extension
%load_ext sql

In [2]:
#2. Connect to (and create) the database file
%sql sqlite:///wk6_joins_exercise_jupyter_sol.db

Connecting to 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

### Create the Customers table

In [3]:
%%sql
-- 1. Create the Customers table
CREATE TABLE Customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL CHECK(length(customer_name) <= 100),
    email TEXT,
    city TEXT
);

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

++
||
++
++

### Create the Orders table

In [4]:
%%sql
-- 2. Create the Orders table
CREATE TABLE Orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
);

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

++
||
++
++

### Import the csv files

In [5]:
import pandas as pd

# 1. Read the CSVs into memory
df_customers = pd.read_csv('customers3.csv')
df_orders = pd.read_csv('orders3.csv')

In [6]:
print(df_customers.columns)
print(df_orders.columns)

Index(['customer_id', 'customer_name', 'email', 'city'], dtype='str')
Index(['order_id', 'customer_id', 'order_date', 'total_amount'], dtype='str')


In [7]:
# Step 1: Push the CSVs as staging tables
# Use the connection we already established with %sql
# Use the --no-index flag to prevent the index from being created
%sql --persist --no-index df_customers
%sql --persist --no-index df_orders

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

Success! Persisted df_customers to the database.

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

Success! Persisted df_orders to the database.

### Append the data

In [8]:
%%sql
-- Move data from staging to the real table
INSERT INTO Customers 
SELECT * FROM df_customers;

-- Move data from staging to the real Orders table
INSERT INTO Orders 
SELECT * FROM df_orders;

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

6 rows affected.

5 rows affected.

++
||
++
++

In [9]:
# Verify ingestion
%sql SELECT * FROM orders LIMIT 5;

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

order_id,customer_id,order_date,total_amount
101,1,Jan 15 2023,150.0
102,3,Jan 20 2023,75.5
103,1,Feb 5 2023,200.0
104,2,Feb 10 2023,125.75
105,None,Feb 15 2023,50.25


In [10]:
%%sql
-- Remove the temporary staging tables
DROP TABLE IF EXISTS df_customers;
DROP TABLE IF EXISTS df_orders;

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

++
||
++
++

In [12]:
%%sql
-- Confirm that both tables were deleted
SELECT name FROM sqlite_master WHERE type='table';

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

name
Customers
Orders


# Run the queries

#### **1. INNER JOIN:** Write a query to display all orders with the customer name who placed them.

In [18]:
%%sql
SELECT c.customer_name, o.order_id, o.total_amount
FROM Customers AS c
INNER JOIN Orders AS o ON c.customer_id = o.customer_id;

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

customer_name,order_id,total_amount
Alice Johnson,101,150.0
Carol Davis,102,75.5
Alice Johnson,103,200.0
Bob Smith,104,125.75


#### **2. LEFT JOIN:** Show all customers and their orders. Include customers who haven't placed any orders.

In [15]:
%%sql
SELECT c.customer_name, o.order_id, o.total_amount
FROM Customers c
LEFT JOIN Orders o ON c.customer_id = o.customer_id;

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

customer_name,order_id,total_amount
Alice Johnson,101,150.0
Alice Johnson,103,200.0
Bob Smith,104,125.75
Carol Davis,102,75.5
David Wilson,None,None
Eve Brown,None,None
Unknown Customer,None,None


#### **3. RIGHT JOIN:** List all orders, including those not associated with any customer.

In [24]:
%%sql
SELECT c.customer_name, o.order_id, o.total_amount
FROM Customers c
RIGHT JOIN Orders o ON c.customer_id = o.customer_id;

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

customer_name,order_id,total_amount
Alice Johnson,101,150.0
Alice Johnson,103,200.0
Bob Smith,104,125.75
Carol Davis,102,75.5
None,105,50.25


#### **4. FULL JOIN:** Display all customers and all orders, showing all possible combinations.

In [25]:
%%sql
SELECT 
    c.customer_name, 
    o.order_id, 
    o.total_amount
FROM Customers c
FULL OUTER JOIN Orders o ON c.customer_id = o.customer_id;

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

customer_name,order_id,total_amount
Alice Johnson,101,150.0
Alice Johnson,103,200.0
Bob Smith,104,125.75
Carol Davis,102,75.5
David Wilson,None,None
Eve Brown,None,None
Unknown Customer,None,None
None,105,50.25


#### **5. CROSS JOIN:** Create a query that shows all possible combinations of customers and cities.

In [23]:
%%sql
SELECT 
    c.customer_name, 
    cities.city
FROM Customers c
CROSS JOIN (
    SELECT DISTINCT city FROM Customers
) AS cities;

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

customer_name,city
Alice Johnson,New York
Alice Johnson,Chicago
Alice Johnson,Los Angeles
Alice Johnson,Miami
Alice Johnson,Boston
Alice Johnson,Unknown
Bob Smith,New York
Bob Smith,Chicago
Bob Smith,Los Angeles
Bob Smith,Miami


In [22]:
%sql SELECT DISTINCT city FROM Customers

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

city
New York
Chicago
Los Angeles
Miami
Boston
Unknown


#### **6. JOIN with WHERE:** Find all customers from 'New York' who have placed orders over $100.

In [20]:
%%sql
SELECT c.customer_name, o.order_id, o.total_amount
FROM Customers c
INNER JOIN Orders o ON c.customer_id = o.customer_id
WHERE c.city = 'New York' 
  AND o.total_amount > 100;

Running query in 'sqlite:///wk6_joins_exercise_jupyter_sol.db'

customer_name,order_id,total_amount
Alice Johnson,101,150.0
Alice Johnson,103,200.0


In [ ]:
# Close the connection
%sql --close sqlite:///wk6_joins_exercise_jupyter_sol.db